# MolAudioNet on Google Colab
## Multi-Modal Molecular AI (Audio + Text + Graph)

**No GPU needed!** This notebook runs fine on free CPU tier.

**What this does:**
- Converts molecules to audio
- Extracts multi-modal features
- Trains ML models for property prediction
- Compares different feature combinations

## Step 1: Install Dependencies (2 minutes)

In [ ]:
# Install required packages
!pip install rdkit scikit-learn pandas numpy matplotlib scipy librosa soundfile -q

# Verify installation
import rdkit
import librosa
import sklearn
print(" All packages installed!")
print(f"RDKit: {rdkit.__version__}")
print(f"Librosa: {librosa.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")

## Step 2: Mount Google Drive (Recommended)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create working directory
import os
!mkdir -p /content/drive/MyDrive/MolAudioNet
os.chdir('/content/drive/MyDrive/MolAudioNet')
print(f"Working directory: {os.getcwd()}")

## Step 3: Upload Scripts

In [ ]:
from google.colab import files

print("Upload these 3 files:")
print("  1. molaudio_pipeline.py")
print("  2. process_moleculenet_standalone.py")
print("  3. feature_combinations.py")
print("\nClick 'Choose Files' below...")

uploaded = files.upload()
print(f"\n Uploaded {len(uploaded)} files")

## Step 4: Upload Dataset

In [ ]:
print("Upload your CSV file (e.g., bbbp.csv, tox21.csv)")
uploaded = files.upload()

# Get the filename
csv_file = list(uploaded.keys())[0]
print(f" Uploaded: {csv_file}")

## Step 5: Quick Test (30 seconds)

In [ ]:
# Test the pipeline on a single molecule
from molaudio_pipeline import MultiModalMolecularAnalyzer, aggregate_features

analyzer = MultiModalMolecularAnalyzer()

# Ibuprofen
smiles = "CC(C)Cc1ccc(C(C)C(=O)O)cc1"
print(f"Testing on: {smiles}\n")

features = analyzer.process_molecule(smiles)

print(" Feature extraction successful!")
print(f"  Audio: {features['audio'].shape}")
print(f"  FFT: {features['fft'].shape}")
print(f"  MFCC: {features['mfcc'].shape}")
print(f"  Text: {len(features['text'])} features")
print(f"  Graph: {len(features['graph'])} features")

combined = aggregate_features(features)
print(f"\n  Combined: {combined.shape[0]} total features")

## Step 6: Process Dataset

Choose your dataset size:
- **Quick test:** 100 molecules (~2 minutes)
- **Medium:** 1000 molecules (~15 minutes)
- **Full BBBP:** 2039 molecules (~25 minutes)

In [ ]:
# Quick test (100 molecules)
!python process_moleculenet_standalone.py --dataset bbbp --csv {csv_file} --limit 100

In [ ]:
# Or full dataset (uncomment to run)
# !python process_moleculenet_standalone.py --dataset bbbp --csv {csv_file}

## Step 7: Compare All Feature Combinations

In [ ]:
!python feature_combinations.py --pkl processed_data/bbbp_features.pkl --compare

## Step 8: Train Your Own Model

In [ ]:
import pickle
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler

# Load features
with open('processed_data/bbbp_features.pkl', 'rb') as f:
    data = pickle.load(f)

# Prepare data
X = np.array(data['combined_features'])
y = np.array(data['labels'])

print(f"Dataset: {X.shape[0]} molecules, {X.shape[1]} features")
print(f"Penetrant: {np.sum(y)} ({100*np.mean(y):.1f}%)\n")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train Random Forest
print("Training Random Forest...")
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1  # Use all CPU cores
)
model.fit(X_train, y_train)
print(" Training complete!\n")

# Evaluate
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print("=" * 60)
print("RESULTS")
print("=" * 60)
print(f"Accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")
print(f"ROC-AUC:  {auc:.3f}")
print("\nLiterature Comparison:")
print("  Random Forest baseline: ~0.72")
print("  Graph Neural Networks:  ~0.90")
print(f"  Our multi-modal:        {accuracy:.3f}")
print("\n" + classification_report(y_test, y_pred, target_names=['Non-penetrant', 'Penetrant']))

## Step 9: Test Individual Feature Combinations

In [ ]:
# Test audio-only features
!python feature_combinations.py --pkl processed_data/bbbp_features.pkl -s audio

In [ ]:
# Test text-only features
!python feature_combinations.py --pkl processed_data/bbbp_features.pkl -s text

In [ ]:
# Test audio + graph (no text)
!python feature_combinations.py --pkl processed_data/bbbp_features.pkl -s audio+graph

## Step 10: Download Results

In [ ]:
# Option 1: Download processed features
files.download('processed_data/bbbp_features.pkl')

In [ ]:
# Option 2: Zip everything and download
!zip -r results.zip processed_data/
files.download('results.zip')

In [ ]:
# Option 3: Already saved to Google Drive (if mounted)
print(" Files saved to: /content/drive/MyDrive/MolAudioNet/")

## Performance Monitoring

In [ ]:
# Check memory usage
!free -h

In [ ]:
# Check disk usage
!df -h

## Next Steps

**You now have:**
-  Processed multi-modal features (audio + text + graph)
-  Trained ML model
-  Performance benchmarks
-  Comparison to literature

**What to do next:**
1. Try different ML models (XGBoost, Neural Networks)
2. Test on Tox21 dataset (toxicity prediction)
3. Process your own custom datasets
4. Optimize hyperparameters
5. Deploy as API or web app

**Questions?**
- Check GOOGLE_COLAB_GUIDE.md for troubleshooting
- See COMPLETE_GUIDE.md for full documentation
- Email: tech@soundofmolecules.com